# Survey-Augmented Knowledge Distillation

Adds ~27 non-invasive survey features (demographics, lifestyle, comorbidities, mental health, diet, family history) as **static features** alongside the 41 daily wearable features.

**Architecture:** Hybrid LSTM — temporal branch (wearable sequences) + static branch (survey MLP), fused before classification head.

Then retrains with knowledge distillation from the CGM teacher.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from pathlib import Path

from config import (RANDOM_STATE, N_FOLDS, LABEL_MAP_4_TO_3, LABEL_MAP_4_TO_2,
                    CLASS_NAMES_4, set_seed, DATASET_PATH)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

FEATURES_DIR = Path('../outputs/features')
SURVEY_DIR = DATASET_PATH / 'survey'

Device: cuda
GPU: NVIDIA GeForce RTX 2050


## Step 1: Extract Survey Features

In [2]:
person_ids = np.load(FEATURES_DIR / 'person_ids.npy')
y = np.load(FEATURES_DIR / 'y.npy')
N = len(person_ids)
print(f'Cohort: {N} participants')

obs = pd.read_csv(SURVEY_DIR / 'observation.csv')
print(f'Observation rows: {len(obs)}, unique persons: {obs["person_id"].nunique()}')

Cohort: 1586 participants


Observation rows: 707126, unique persons: 2280


In [3]:
# Pivot: person_id x observation_source_value -> value_as_number
obs_pivot = obs.pivot_table(
    index='person_id',
    columns='observation_source_value',
    values='value_as_number',
    aggfunc='first'
)
print(f'Pivot shape: {obs_pivot.shape}')

SENTINEL_VALUES = {777.0, 555.0, 888.0, 999.0, 99.0}

def get_feat(pid, col_prefix):
    matching = [c for c in obs_pivot.columns if c.startswith(col_prefix + ',') or c == col_prefix]
    if not matching or pid not in obs_pivot.index:
        return np.nan
    val = obs_pivot.loc[pid, matching[0]]
    if pd.isna(val) or val in SENTINEL_VALUES:
        return np.nan
    return float(val)

Pivot shape: (2280, 292)


In [4]:
# 27 non-invasive survey features
# EXCLUDED: mhterm_dm2, mhterm_predm, cmtrt_*, mh_a1c (label leakage)
SURVEY_FEATURES = [
    # Demographics
    ('age',              'cage'),
    ('education_years',  'years_of_education'),
    # Lifestyle — smoking
    ('smoking_ever',     'susmkncf'),
    ('smoking_now',      'susmkcdur'),
    # Lifestyle — alcohol
    ('alcohol_ever',     'sualckncf'),
    # Mental health
    ('cesd_score',       'cestl'),
    ('sleep_restless',   'ces7'),
    ('paid_score',       'paidscore'),
    # Medications (non-diabetes)
    ('sleeping_pills',   'cm_slp'),
    # Diet
    ('diet_score',       'dietscore'),
    ('diet_fast_food',   'diet1'),
    ('diet_beans',       'diet5'),
    ('diet_regular_food','diet6'),
    ('diet_desserts',    'diet7'),
    ('diet_fats',        'diet8'),
    # Family history
    ('fh_diabetes_parent', 'fh_dm2pt'),
    ('fh_diabetes_sibling','fh_dm2sb'),
    # Self-reported comorbidities
    ('has_hypertension', 'mhoccur_hbp'),
    ('has_obesity',      'mhoccur_obs'),
    ('has_high_cholesterol', 'mhoccur_clsh'),
    ('has_heart_attack', 'mhoccur_mi'),
    ('has_stroke',       'mhoccur_strk'),
    ('has_kidney_problems', 'mhoccur_rnl'),
    ('has_circulation',  'mhoccur_circ'),
    # Vision difficulty
    ('vision_difficulty','via1'),
    # Food insecurity
    ('food_insecurity_1','pxfi1'),
    ('food_insecurity_2','pxfi2'),
]

SURVEY_FEATURE_NAMES = [name for name, _ in SURVEY_FEATURES]
N_SURVEY = len(SURVEY_FEATURE_NAMES)
print(f'Extracting {N_SURVEY} survey features...')

Extracting 27 survey features...


In [5]:
X_survey = np.full((N, N_SURVEY), np.nan, dtype=np.float32)

for i, pid in enumerate(person_ids):
    for j, (name, col_prefix) in enumerate(SURVEY_FEATURES):
        X_survey[i, j] = get_feat(int(pid), col_prefix)

# Coverage report
print(f'Survey feature coverage (out of {N}):\n')
for j, name in enumerate(SURVEY_FEATURE_NAMES):
    valid = np.sum(~np.isnan(X_survey[:, j]))
    print(f'  {name:25s}: {valid:5d}/{N}  ({valid/N*100:.1f}%)')

print(f'\nOverall coverage: {np.mean(~np.isnan(X_survey))*100:.1f}%')

np.save(FEATURES_DIR / 'X_survey.npy', X_survey)
print(f'\nSaved X_survey.npy: {X_survey.shape}')

Survey feature coverage (out of 1586):

  age                      :  1586/1586  (100.0%)
  education_years          :  1579/1586  (99.6%)
  smoking_ever             :  1580/1586  (99.6%)
  smoking_now              :   479/1586  (30.2%)
  alcohol_ever             :  1574/1586  (99.2%)
  cesd_score               :  1583/1586  (99.8%)
  sleep_restless           :  1579/1586  (99.6%)
  paid_score               :  1556/1586  (98.1%)
  sleeping_pills           :  1551/1586  (97.8%)
  diet_score               :   481/1586  (30.3%)
  diet_fast_food           :  1541/1586  (97.2%)
  diet_beans               :  1539/1586  (97.0%)
  diet_regular_food        :  1540/1586  (97.1%)
  diet_desserts            :  1539/1586  (97.0%)
  diet_fats                :  1539/1586  (97.0%)
  fh_diabetes_parent       :  1438/1586  (90.7%)
  fh_diabetes_sibling      :  1426/1586  (89.9%)
  has_hypertension         :  1580/1586  (99.6%)
  has_obesity              :  1559/1586  (98.3%)
  has_high_cholesterol     :

## Step 2: Load Wearable Data + Teacher OOF

In [6]:
X_orig = np.load(FEATURES_DIR / 'X.npy')
X_c = np.load(FEATURES_DIR / 'X_option_c.npy')
lengths = np.load(FEATURES_DIR / 'lengths.npy')
teacher_oof = np.load(FEATURES_DIR / 'teacher_oof_proba.npy')

X_wear = np.concatenate([X_orig, X_c], axis=2)  # (1586, 14, 41)

NF_WEAR = X_wear.shape[2]
SEQ_LEN = X_wear.shape[1]

print(f'X_wear:      {X_wear.shape}')
print(f'X_survey:    {X_survey.shape}')
print(f'Teacher OOF: {teacher_oof.shape}')
print(f'Classes:     {np.bincount(y)}')

X_wear:      (1586, 14, 41)
X_survey:    (1586, 27)
Teacher OOF: (1586, 4)
Classes:     [560 410 446 170]


## Step 3: Hybrid LSTM Model

In [7]:
class HybridLSTMAttn(nn.Module):
    """LSTM+Attention for temporal features + static MLP for survey data, fused before head."""
    def __init__(self, seq_input_size, static_size, hidden=64, layers=2, drop=0.4):
        super().__init__()
        self.lstm = nn.LSTM(seq_input_size, hidden, layers,
                            dropout=drop if layers > 1 else 0,
                            batch_first=True)
        self.attn = nn.Linear(hidden, 1)
        self.static_net = nn.Sequential(
            nn.Linear(static_size, 32), nn.ReLU(), nn.Dropout(drop / 2),
        )
        fused_size = hidden + 32
        self.head = nn.Sequential(
            nn.BatchNorm1d(fused_size), nn.Dropout(drop),
            nn.Linear(fused_size, 64), nn.ReLU(),
            nn.Dropout(drop / 2), nn.Linear(64, 4),
        )

    def forward(self, x_seq, x_static, lens=None):
        if lens is not None:
            pk = nn.utils.rnn.pack_padded_sequence(
                x_seq, lens.cpu(), batch_first=True, enforce_sorted=False)
            out, _ = self.lstm(pk)
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        else:
            out, _ = self.lstm(x_seq)
        scores = self.attn(out).squeeze(-1)
        if lens is not None:
            mask = (torch.arange(out.size(1), device=out.device).unsqueeze(0)
                    >= lens.unsqueeze(1).to(out.device))
            scores = scores.masked_fill(mask, float('-inf'))
        w = torch.softmax(scores, dim=1).unsqueeze(-1)
        temporal = (out * w).sum(dim=1)
        static = self.static_net(x_static)
        fused = torch.cat([temporal, static], dim=1)
        return self.head(fused)

In [8]:
def compute_metrics(y, oof_proba, label=''):
    pred = np.argmax(oof_proba, axis=1)
    acc = accuracy_score(y, pred)
    auc4 = roc_auc_score(y, oof_proba, multi_class='ovr', average='macro')
    y3 = np.array([LABEL_MAP_4_TO_3[i] for i in y])
    p3 = np.zeros((len(y), 3))
    p3[:, 0] = oof_proba[:, 0]; p3[:, 1] = oof_proba[:, 1]
    p3[:, 2] = oof_proba[:, 2] + oof_proba[:, 3]
    auc3 = roc_auc_score(y3, p3, multi_class='ovr', average='macro')
    y2 = np.array([LABEL_MAP_4_TO_2[i] for i in y])
    auc2 = roc_auc_score(y2, oof_proba[:, 1] + oof_proba[:, 2] + oof_proba[:, 3])
    print(f'\n  {label}: acc={acc*100:.1f}%  4-AUC={auc4:.4f}  3-AUC={auc3:.4f}  2-AUC={auc2:.4f}')
    print(classification_report(y, pred, target_names=CLASS_NAMES_4, digits=3))
    return acc, auc4, auc3, auc2

def _class_weights(y_train):
    cc = np.bincount(y_train, minlength=4).astype(float)
    cw = 1.0 / (cc + 1e-6); cw = cw / cw.sum() * 4
    return torch.tensor(cw, dtype=torch.float32).to(device)

def _prep_fold_hybrid(X_seq, X_static, y, lengths, tr, va):
    nf_seq = X_seq.shape[2]
    nf_stat = X_static.shape[1]
    ntr, nva = len(tr), len(va)
    # Temporal
    Xtr_s = X_seq[tr].reshape(-1, nf_seq).copy()
    Xva_s = X_seq[va].reshape(-1, nf_seq).copy()
    imp_s = SimpleImputer(strategy='median')
    Xtr_s = imp_s.fit_transform(Xtr_s)
    Xva_s = imp_s.transform(Xva_s)
    sc_s = StandardScaler()
    Xtr_s = sc_s.fit_transform(Xtr_s)
    Xva_s = sc_s.transform(Xva_s)
    # Static
    Xtr_st = X_static[tr].copy()
    Xva_st = X_static[va].copy()
    imp_st = SimpleImputer(strategy='median')
    Xtr_st = imp_st.fit_transform(Xtr_st)
    Xva_st = imp_st.transform(Xva_st)
    sc_st = StandardScaler()
    Xtr_st = sc_st.fit_transform(Xtr_st)
    Xva_st = sc_st.transform(Xva_st)
    return (
        torch.tensor(Xtr_s.reshape(ntr, SEQ_LEN, nf_seq).astype(np.float32)),
        torch.tensor(Xtr_st.astype(np.float32)),
        torch.tensor(y[tr], dtype=torch.long),
        torch.tensor(lengths[tr], dtype=torch.long),
        torch.tensor(Xva_s.reshape(nva, SEQ_LEN, nf_seq).astype(np.float32)),
        torch.tensor(Xva_st.astype(np.float32)),
        torch.tensor(y[va], dtype=torch.long),
        torch.tensor(lengths[va], dtype=torch.long),
    )

def distillation_loss(student_logits, hard_labels, teacher_probs, T, alpha, cls_crit):
    l_hard = cls_crit(student_logits, hard_labels)
    student_log_soft = F.log_softmax(student_logits / T, dim=1)
    teacher_soft = torch.tensor(teacher_probs, dtype=torch.float32, device=student_logits.device)
    teacher_log = torch.log(teacher_soft + 1e-8)
    teacher_tempered = F.softmax(teacher_log / T, dim=1)
    l_soft = F.kl_div(student_log_soft, teacher_tempered, reduction='batchmean') * (T * T)
    return alpha * l_hard + (1 - alpha) * l_soft

## Step 4: Train Hybrid Student with Distillation

Using the **original CGM teacher** OOF predictions (4-AUC=0.7472) to distill into the hybrid student (wearable + survey).

Best previous config: T=2, alpha=0.3

In [9]:
def run_hybrid_student_cv(X_seq, X_static, y, lengths, teacher_oof_probs,
                          T=2.0, alpha=0.3, label='HybridStudent'):
    set_seed()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros((len(y), 4))
    nf_seq = X_seq.shape[2]
    nf_stat = X_static.shape[1]

    print(f'\n[{label}]  T={T}  alpha={alpha}  seq_feat={nf_seq}  static_feat={nf_stat}')

    for fold, (tr, va) in enumerate(skf.split(X_seq, y)):
        print(f'  Fold {fold+1}/{N_FOLDS}...', end=' ')
        (Xt_seq, Xt_stat, yt, lt,
         Xv_seq, Xv_stat, yv, lv) = _prep_fold_hybrid(X_seq, X_static, y, lengths, tr, va)
        Xv_seq, Xv_stat, yv = Xv_seq.to(device), Xv_stat.to(device), yv.to(device)

        teacher_tr = teacher_oof_probs[tr]
        loader = DataLoader(
            TensorDataset(Xt_seq, Xt_stat, yt, lt,
                          torch.tensor(teacher_tr, dtype=torch.float32)),
            batch_size=64, shuffle=True)
        wt = _class_weights(y[tr])

        set_seed()
        model = HybridLSTMAttn(seq_input_size=nf_seq, static_size=nf_stat).to(device)
        cls_crit = nn.CrossEntropyLoss(weight=wt, label_smoothing=0.1)
        opt = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=7, factor=0.5)

        best_vl, pat, best_state = float('inf'), 0, None
        for ep in range(100):
            model.train()
            for Xb_seq, Xb_stat, yb, lb, tb in loader:
                Xb_seq, Xb_stat, yb = Xb_seq.to(device), Xb_stat.to(device), yb.to(device)
                opt.zero_grad()
                logits = model(Xb_seq, Xb_stat, lb)
                loss = distillation_loss(logits, yb, tb.numpy(), T, alpha, cls_crit)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            model.eval()
            with torch.no_grad():
                vl = cls_crit(model(Xv_seq, Xv_stat, lv), yv).item()
            sched.step(vl)
            if vl < best_vl:
                best_vl = vl; pat = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                pat += 1
                if pat >= 20:
                    print(f'early stop ep {ep+1}', end=' ')
                    break

        model.load_state_dict(best_state); model.eval()
        with torch.no_grad():
            oof_proba[va] = torch.softmax(model(Xv_seq, Xv_stat, lv), dim=1).cpu().numpy()
        print(f'done (best_vl={best_vl:.4f})')

    acc, auc4, auc3, auc2 = compute_metrics(y, oof_proba, label)
    np.save(FEATURES_DIR / f'oof_{label.replace(" ", "_")}.npy', oof_proba)
    return acc, auc4, auc3, auc2, oof_proba

In [10]:
# Run with best previous config: T=2, alpha=0.3
acc, auc4, auc3, auc2, hybrid_oof = run_hybrid_student_cv(
    X_wear, X_survey, y, lengths, teacher_oof,
    T=2.0, alpha=0.3, label='hybrid_T2_a0.3')


[hybrid_T2_a0.3]  T=2.0  alpha=0.3  seq_feat=41  static_feat=27
  Fold 1/5... 

early stop ep 42 done (best_vl=1.2257)
  Fold 2/5... 

early stop ep 41 done (best_vl=1.3121)
  Fold 3/5... 

early stop ep 43 done (best_vl=1.2747)
  Fold 4/5... 

early stop ep 37 done (best_vl=1.2847)
  Fold 5/5... 

early stop ep 36 done (best_vl=1.2631)

  hybrid_T2_a0.3: acc=43.5%  4-AUC=0.7163  3-AUC=0.7166  2-AUC=0.7376
              precision    recall  f1-score   support

     healthy      0.573     0.627     0.598       560
 prediabetes      0.406     0.268     0.323       410
    oral_med      0.435     0.253     0.320       446
     insulin      0.262     0.682     0.379       170

    accuracy                          0.435      1586
   macro avg      0.419     0.458     0.405      1586
weighted avg      0.457     0.435     0.425      1586



## Step 5: Sweep T and Alpha

In [11]:
results = {}
for T in [2.0, 3.0, 4.0]:
    for alpha in [0.3, 0.5, 0.7]:
        label = f'hybrid_T{T:.0f}_a{alpha}'
        acc, auc4, auc3, auc2, oof = run_hybrid_student_cv(
            X_wear, X_survey, y, lengths, teacher_oof,
            T=T, alpha=alpha, label=label)
        results[(T, alpha)] = dict(acc=acc, auc4=auc4, auc3=auc3, auc2=auc2)
        print('-' * 60)


[hybrid_T2_a0.3]  T=2.0  alpha=0.3  seq_feat=41  static_feat=27
  Fold 1/5... 

early stop ep 42 done (best_vl=1.2257)
  Fold 2/5... 

early stop ep 41 done (best_vl=1.3121)
  Fold 3/5... 

early stop ep 43 done (best_vl=1.2747)
  Fold 4/5... 

early stop ep 37 done (best_vl=1.2847)
  Fold 5/5... 

...

early stop ep 40 done (best_vl=1.2128)
  Fold 2/5... 

early stop ep 36 done (best_vl=1.3217)
  Fold 3/5... 

early stop ep 34 done (best_vl=1.2772)
  Fold 4/5... 

early stop ep 31 done (best_vl=1.2879)
  Fold 5/5... 

early stop ep 33 done (best_vl=1.2800)

  hybrid_T4_a0.7: acc=41.9%  4-AUC=0.7115  3-AUC=0.7183  2-AUC=0.7469
              precision    recall  f1-score   support

     healthy      0.587     0.568     0.577       560
 prediabetes      0.367     0.312     0.337       410
    oral_med      0.420     0.242     0.307       446
     insulin      0.251     0.647     0.362       170

    accuracy                          0.419      1586
   macro avg      0.406     0.442     0.396      1586
weighted avg      0.447     0.419     0.416      1586

------------------------------------------------------------


## Step 6: Ensemble — Hybrid + Previous Best

In [12]:
prev_best_oof = np.load(FEATURES_DIR / 'oof_T2_a0.3.npy')  # wearable-only distilled

best_key = max(results, key=lambda k: results[k]['auc4'])
best_hybrid_label = f'hybrid_T{best_key[0]:.0f}_a{best_key[1]}'
best_hybrid_oof = np.load(FEATURES_DIR / f'oof_{best_hybrid_label}.npy')

print('Ensemble: w * hybrid + (1-w) * prev_wearable_distilled\n')
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ens = w * best_hybrid_oof + (1 - w) * prev_best_oof
    compute_metrics(y, ens, f'Ensemble w={w:.1f}')

Ensemble: w * hybrid + (1-w) * prev_wearable_distilled


  Ensemble w=0.3: acc=40.6%  4-AUC=0.7018  3-AUC=0.7010  2-AUC=0.7103
              precision    recall  f1-score   support

     healthy      0.546     0.543     0.544       560
 prediabetes      0.385     0.317     0.348       410
    oral_med      0.441     0.200     0.275       446
     insulin      0.247     0.712     0.367       170

    accuracy                          0.406      1586
   macro avg      0.405     0.443     0.383      1586
weighted avg      0.443     0.406     0.399      1586


  Ensemble w=0.4: acc=41.2%  4-AUC=0.7053  3-AUC=0.7053  2-AUC=0.7177
              precision    recall  f1-score   support

     healthy      0.554     0.564     0.559       560
 prediabetes      0.388     0.307     0.343       410
    oral_med      0.446     0.204     0.280       446
     insulin      0.248     0.712     0.368       170

    accuracy                          0.412      1586
   macro avg      0.409     0.447     0.3


  Ensemble w=0.5: acc=41.7%  4-AUC=0.7084  3-AUC=0.7090  2-AUC=0.7242
              precision    recall  f1-score   support

     healthy      0.553     0.575     0.564       560
 prediabetes      0.384     0.295     0.334       410
    oral_med      0.460     0.220     0.297       446
     insulin      0.252     0.706     0.372       170

    accuracy                          0.417      1586
   macro avg      0.412     0.449     0.392      1586
weighted avg      0.451     0.417     0.409      1586


  Ensemble w=0.6: acc=42.6%  4-AUC=0.7109  3-AUC=0.7123  2-AUC=0.7304
              precision    recall  f1-score   support

     healthy      0.566     0.593     0.579       560
 prediabetes      0.395     0.300     0.341       410
    oral_med      0.452     0.233     0.308       446
     insulin      0.255     0.688     0.373       170

    accuracy                          0.426      1586
   macro avg      0.417     0.454     0.400      1586
weighted avg      0.456     0.426     0.419


  Ensemble w=0.7: acc=43.0%  4-AUC=0.7130  3-AUC=0.7151  2-AUC=0.7357
              precision    recall  f1-score   support

     healthy      0.568     0.593     0.580       560
 prediabetes      0.396     0.298     0.340       410
    oral_med      0.455     0.251     0.324       446
     insulin      0.259     0.682     0.375       170

    accuracy                          0.430      1586
   macro avg      0.420     0.456     0.405      1586
weighted avg      0.459     0.430     0.424      1586



## Final Comparison

In [13]:
print(f"{'Model':<45} {'Acc':>7} {'4-AUC':>7} {'3-AUC':>7} {'2-AUC':>7}")
print('=' * 75)
print(f"{'Option C LSTM (no distillation)':<45} {'33.9%':>7} {'0.6725':>7} {'0.6706':>7} {'0.6632':>7}")
print(f"{'Distilled T2a0.3 (wearable only)':<45} {'37.6%':>7} {'0.6879':>7} {'0.6858':>7} {'0.6846':>7}")
print(f"{'Teacher (wearable+CGM)':<45} {'45.4%':>7} {'0.7472':>7} {'—':>7} {'—':>7}")
print()
print('Hybrid Students (wearable + 27 survey features, distilled):')
print('-' * 75)

best_key = max(results, key=lambda k: results[k]['auc4'])
for (T, alpha), r in sorted(results.items()):
    tag = '  << BEST' if (T, alpha) == best_key else ''
    name = f'  T={T:.0f} alpha={alpha}'
    print(f'{name:<45} {r["acc"]*100:>6.1f}% {r["auc4"]:>7.4f} {r["auc3"]:>7.4f} {r["auc2"]:>7.4f}{tag}')

best_r = results[best_key]
print(f'\nBest hybrid: T={best_key[0]:.0f}, alpha={best_key[1]}')
print(f'  vs prev best (0.6879):   {best_r["auc4"]-0.6879:+.4f}')
print(f'  vs Teacher (0.7472):     {best_r["auc4"]-0.7472:+.4f}')

Model                                             Acc   4-AUC   3-AUC   2-AUC
Option C LSTM (no distillation)                 33.9%  0.6725  0.6706  0.6632
Distilled T2a0.3 (wearable only)                37.6%  0.6879  0.6858  0.6846
Teacher (wearable+CGM)                          45.4%  0.7472       —       —

Hybrid Students (wearable + 27 survey features, distilled):
---------------------------------------------------------------------------
  T=2 alpha=0.3                                 43.5%  0.7163  0.7166  0.7376
  T=2 alpha=0.5                                 43.3%  0.7167  0.7213  0.7474  << BEST
  T=2 alpha=0.7                                 42.0%  0.7122  0.7192  0.7497
  T=3 alpha=0.3                                 43.5%  0.7163  0.7169  0.7376
  T=3 alpha=0.5                                 43.6%  0.7165  0.7217  0.7475
  T=3 alpha=0.7                                 42.7%  0.7133  0.7210  0.7514
  T=4 alpha=0.3                                 43.2%  0.7164  0.7170  0.7